# DQN Model Evaluation - Compute IoU Metrics

This notebook evaluates the trained DQN model on 200 test images and computes:
- Overall IoU
- Thin Cloud IoU
- All performance metrics (Accuracy, Precision, Recall, F1)

**Run this in Google Colab with GPU runtime.**

## 1. Setup and Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install stable-baselines3 gymnasium rasterio scikit-learn -q

print("✅ Setup complete!")

## 2. Configuration and Imports

In [ ]:
import os
import glob
import numpy as np
import rasterio
from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score, accuracy_score
from stable_baselines3 import DQN
import gymnasium as gym
from gymnasium import spaces

# Paths - UPDATE THESE IF NEEDED
DATA_DIR = '/content/drive/MyDrive/Colab_Data/cloudsen12_processed_1000'
DQN_MODEL_PATH = '/content/drive/MyDrive/Colab_Data/dqn_thin_cloud/dqn_thin_cloud_100000_steps.zip'

# Verify paths exist
print(f"Data directory exists: {os.path.exists(DATA_DIR)}")
print(f"DQN model exists: {os.path.exists(DQN_MODEL_PATH)}")

# Count files
image_files = sorted(glob.glob(f'{DATA_DIR}/*_image.tif'))
mask_files = sorted(glob.glob(f'{DATA_DIR}/*_mask.tif'))
print(f"\n📂 Found {len(image_files)} images and {len(mask_files)} masks")

## 3. Define Environment (Same as Training)

In [ ]:
class ThinCloudDetectionEnvDiscrete(gym.Env):
    """
    Discrete action space environment for DQN thin cloud detection.
    15 discrete actions: combinations of threshold adjustments and boosts.
    """
    
    def __init__(self, cnn_prob, ground_truth, patch_size=64):
        super().__init__()
        
        self.cnn_prob = cnn_prob.astype(np.float32)
        self.ground_truth = ground_truth
        self.patch_size = patch_size
        self.h, self.w = cnn_prob.shape
        
        # Create thin cloud mask (class 2)
        self.thin_cloud_mask = (ground_truth == 2)
        self.cloud_mask = (ground_truth >= 1)  # All clouds
        
        # Grid of patches
        self.n_patches_h = self.h // patch_size
        self.n_patches_w = self.w // patch_size
        self.total_patches = self.n_patches_h * self.n_patches_w
        
        # 15 discrete actions: 5 thresholds × 3 boosts
        self.action_space = spaces.Discrete(15)
        
        # Action mapping
        self.threshold_values = [-0.20, -0.10, 0.00, 0.10, 0.20]
        self.boost_values = [0.00, 0.25, 0.50]
        
        # 20-dim observation space
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32
        )
        
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        
    def _get_action_values(self, action):
        """Convert discrete action to threshold and boost values."""
        thresh_idx = action // 3
        boost_idx = action % 3
        return self.threshold_values[thresh_idx], self.boost_values[boost_idx]
    
    def _get_patch_coords(self, patch_idx):
        """Get patch coordinates."""
        row = patch_idx // self.n_patches_w
        col = patch_idx % self.n_patches_w
        y1 = row * self.patch_size
        y2 = y1 + self.patch_size
        x1 = col * self.patch_size
        x2 = x1 + self.patch_size
        return y1, y2, x1, x2
    
    def _get_observation(self):
        """Extract 20-feature observation for current patch."""
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        patch_prob = self.cnn_prob[y1:y2, x1:x2]
        patch_thin = self.thin_cloud_mask[y1:y2, x1:x2]
        
        # CNN probability statistics
        prob_mean = np.mean(patch_prob)
        prob_std = np.std(patch_prob)
        prob_max = np.max(patch_prob)
        prob_min = np.min(patch_prob)
        
        # Probability distribution
        prob_median = np.median(patch_prob)
        prob_q25 = np.percentile(patch_prob, 25)
        prob_q75 = np.percentile(patch_prob, 75)
        
        # Edge/gradient features
        grad_y = np.abs(np.diff(patch_prob, axis=0)).mean()
        grad_x = np.abs(np.diff(patch_prob, axis=1)).mean()
        
        # Thin cloud indicators
        thin_ratio = np.mean(patch_thin)
        uncertain_ratio = np.mean((patch_prob > 0.3) & (patch_prob < 0.7))
        
        # Spatial context
        row_norm = (self.current_patch // self.n_patches_w) / self.n_patches_h
        col_norm = (self.current_patch % self.n_patches_w) / self.n_patches_w
        
        # High probability region
        high_prob_ratio = np.mean(patch_prob > 0.5)
        low_prob_ratio = np.mean(patch_prob < 0.3)
        
        # Texture features
        local_var = np.var(patch_prob)
        
        # Additional features
        prob_range = prob_max - prob_min
        skewness = ((patch_prob - prob_mean) ** 3).mean() / (prob_std ** 3 + 1e-8)
        
        obs = np.array([
            prob_mean, prob_std, prob_max, prob_min,
            prob_median, prob_q25, prob_q75,
            grad_y, grad_x,
            thin_ratio, uncertain_ratio,
            row_norm, col_norm,
            high_prob_ratio, low_prob_ratio,
            local_var, prob_range, skewness,
            0.0, 0.0  # Padding to 20 features
        ], dtype=np.float32)
        
        return obs
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        return self._get_observation(), {}
    
    def step(self, action):
        threshold_delta, thin_boost = self._get_action_values(action)
        
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        # Apply refinement
        patch = self.refined_prob[y1:y2, x1:x2].copy()
        
        # Lower threshold = more sensitive (add negative delta to prob)
        patch = patch - threshold_delta
        
        # Apply thin cloud boost to uncertain regions
        uncertain_mask = (patch > 0.2) & (patch < 0.6)
        patch[uncertain_mask] += thin_boost
        
        patch = np.clip(patch, 0, 1)
        self.refined_prob[y1:y2, x1:x2] = patch
        
        # Move to next patch
        self.current_patch += 1
        done = self.current_patch >= self.total_patches
        
        # Compute reward
        reward = self._compute_reward(y1, y2, x1, x2)
        
        if done:
            obs = np.zeros(20, dtype=np.float32)
        else:
            obs = self._get_observation()
        
        return obs, reward, done, False, {}
    
    def _compute_reward(self, y1, y2, x1, x2):
        """Multi-objective reward: 70% thin cloud IoU + 30% F1."""
        patch_pred = (self.refined_prob[y1:y2, x1:x2] > 0.5).flatten()
        patch_gt_cloud = self.cloud_mask[y1:y2, x1:x2].flatten()
        patch_gt_thin = self.thin_cloud_mask[y1:y2, x1:x2].flatten()
        
        # Thin cloud IoU
        thin_intersection = np.sum(patch_pred & patch_gt_thin)
        thin_union = np.sum(patch_pred | patch_gt_thin)
        if thin_union > 0:
            thin_iou = thin_intersection / (thin_union + 1e-8)
        else:
            thin_iou = 0.5
        
        # F1 score
        tp = np.sum(patch_pred & patch_gt_cloud)
        fp = np.sum(patch_pred & ~patch_gt_cloud)
        fn = np.sum(~patch_pred & patch_gt_cloud)
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
        reward = 0.7 * thin_iou + 0.3 * f1
        return reward
    
    def get_refined_mask(self):
        """Return the refined binary mask."""
        return (self.refined_prob > 0.5).astype(np.uint8)

print("✅ Environment defined!")

## 4. Load DQN Model and Test Data

In [ ]:
# Load DQN model
print("Loading DQN model...")
dqn_model = DQN.load(DQN_MODEL_PATH)
print("✅ DQN model loaded!")

# Load test data (last 200 images = 20% of 1000)
TRAIN_SPLIT = 0.8
split_idx = int(TRAIN_SPLIT * len(image_files))

test_images = image_files[split_idx:]
test_masks = mask_files[split_idx:]

print(f"\n📊 Test set: {len(test_images)} images")

## 5. CNN Baseline Function

In [ ]:
# Try to use s2cloudless, fallback to simple threshold
try:
    from s2cloudless import S2PixelCloudDetector
    cloud_detector = S2PixelCloudDetector(threshold=0.4, average_over=4, dilation_size=2)
    USE_S2CLOUDLESS = True
    print("✅ Using s2cloudless for baseline")
except ImportError:
    !pip install s2cloudless -q
    from s2cloudless import S2PixelCloudDetector
    cloud_detector = S2PixelCloudDetector(threshold=0.4, average_over=4, dilation_size=2)
    USE_S2CLOUDLESS = True
    print("✅ Installed and using s2cloudless for baseline")

def get_cnn_probability(image_path):
    """Get CNN cloud probability map."""
    with rasterio.open(image_path) as src:
        bands = src.read()  # Shape: (13, H, W)
    
    # Normalize bands to 0-1
    bands = bands.astype(np.float32) / 10000.0
    bands = np.clip(bands, 0, 1)
    
    # Reshape for s2cloudless: (1, H, W, 13)
    bands_reshaped = np.transpose(bands, (1, 2, 0))[np.newaxis, ...]
    
    # Get probability
    prob = cloud_detector.get_cloud_probability_maps(bands_reshaped)[0]
    
    return prob.astype(np.float32)

print("✅ CNN baseline function ready!")

## 6. Evaluate DQN on Test Set

In [ ]:
# Metrics accumulators
baseline_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}
dqn_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}

# Thin cloud specific
baseline_thin = {'tp': 0, 'total': 0}
dqn_thin = {'tp': 0, 'total': 0}

# IoU accumulators
baseline_iou_sum = 0
dqn_iou_sum = 0
baseline_thin_iou_sum = 0
dqn_thin_iou_sum = 0
n_images = 0

print("Evaluating DQN on test set...")
print("="*60)

for i, (img_path, mask_path) in enumerate(zip(test_images, test_masks)):
    if (i + 1) % 20 == 0:
        print(f"Processing image {i+1}/{len(test_images)}...")
    
    try:
        # Load ground truth
        with rasterio.open(mask_path) as src:
            gt = src.read(1)
        
        # Get CNN probability
        cnn_prob = get_cnn_probability(img_path)
        
        # Baseline prediction (threshold 0.5)
        baseline_pred = (cnn_prob > 0.5).astype(np.uint8)
        
        # DQN refinement
        env = ThinCloudDetectionEnvDiscrete(cnn_prob, gt)
        obs, _ = env.reset()
        
        done = False
        while not done:
            action, _ = dqn_model.predict(obs, deterministic=True)
            obs, _, done, _, _ = env.step(action)
        
        dqn_pred = env.get_refined_mask()
        
        # Ground truth masks
        gt_cloud = (gt >= 1)  # All clouds
        gt_thin = (gt == 2)   # Thin clouds only
        
        # Flatten for metrics
        baseline_flat = baseline_pred.flatten().astype(bool)
        dqn_flat = dqn_pred.flatten().astype(bool)
        gt_cloud_flat = gt_cloud.flatten()
        gt_thin_flat = gt_thin.flatten()
        
        # Overall metrics
        baseline_metrics['tp'] += np.sum(baseline_flat & gt_cloud_flat)
        baseline_metrics['fp'] += np.sum(baseline_flat & ~gt_cloud_flat)
        baseline_metrics['tn'] += np.sum(~baseline_flat & ~gt_cloud_flat)
        baseline_metrics['fn'] += np.sum(~baseline_flat & gt_cloud_flat)
        
        dqn_metrics['tp'] += np.sum(dqn_flat & gt_cloud_flat)
        dqn_metrics['fp'] += np.sum(dqn_flat & ~gt_cloud_flat)
        dqn_metrics['tn'] += np.sum(~dqn_flat & ~gt_cloud_flat)
        dqn_metrics['fn'] += np.sum(~dqn_flat & gt_cloud_flat)
        
        # Thin cloud recall
        if np.sum(gt_thin_flat) > 0:
            baseline_thin['tp'] += np.sum(baseline_flat & gt_thin_flat)
            baseline_thin['total'] += np.sum(gt_thin_flat)
            dqn_thin['tp'] += np.sum(dqn_flat & gt_thin_flat)
            dqn_thin['total'] += np.sum(gt_thin_flat)
        
        # IoU calculation
        baseline_intersection = np.sum(baseline_flat & gt_cloud_flat)
        baseline_union = np.sum(baseline_flat | gt_cloud_flat)
        dqn_intersection = np.sum(dqn_flat & gt_cloud_flat)
        dqn_union = np.sum(dqn_flat | gt_cloud_flat)
        
        if baseline_union > 0:
            baseline_iou_sum += baseline_intersection / baseline_union
        if dqn_union > 0:
            dqn_iou_sum += dqn_intersection / dqn_union
        
        # Thin cloud IoU
        baseline_thin_inter = np.sum(baseline_flat & gt_thin_flat)
        baseline_thin_union = np.sum(baseline_flat | gt_thin_flat)
        dqn_thin_inter = np.sum(dqn_flat & gt_thin_flat)
        dqn_thin_union = np.sum(dqn_flat | gt_thin_flat)
        
        if baseline_thin_union > 0:
            baseline_thin_iou_sum += baseline_thin_inter / baseline_thin_union
        if dqn_thin_union > 0:
            dqn_thin_iou_sum += dqn_thin_inter / dqn_thin_union
        
        n_images += 1
        
    except Exception as e:
        print(f"Error on image {i}: {e}")
        continue

print(f"\n✅ Evaluated {n_images} images successfully!")

## 7. Compute and Display Final Metrics

In [ ]:
def compute_all_metrics(m):
    """Compute all metrics from confusion matrix components."""
    tp, fp, tn, fn = m['tp'], m['fp'], m['tn'], m['fn']
    total = tp + fp + tn + fn
    
    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    
    return accuracy, precision, recall, f1, iou

# Compute metrics
b_acc, b_prec, b_rec, b_f1, b_iou = compute_all_metrics(baseline_metrics)
d_acc, d_prec, d_rec, d_f1, d_iou = compute_all_metrics(dqn_metrics)

# Thin cloud recall
b_thin_recall = baseline_thin['tp'] / baseline_thin['total'] if baseline_thin['total'] > 0 else 0
d_thin_recall = dqn_thin['tp'] / dqn_thin['total'] if dqn_thin['total'] > 0 else 0

# Average IoU across images
b_avg_iou = baseline_iou_sum / n_images if n_images > 0 else 0
d_avg_iou = dqn_iou_sum / n_images if n_images > 0 else 0

b_avg_thin_iou = baseline_thin_iou_sum / n_images if n_images > 0 else 0
d_avg_thin_iou = dqn_thin_iou_sum / n_images if n_images > 0 else 0

# Print results
print("\n" + "="*70)
print("📊 DQN EVALUATION RESULTS - 200 TEST IMAGES")
print("="*70)

print("\n📈 OVERALL CLOUD DETECTION METRICS:")
print("-"*70)
print(f"{'Metric':<25} {'CNN Baseline':>15} {'DQN (100k)':>15} {'Change':>12}")
print("-"*70)
print(f"{'Accuracy':<25} {b_acc*100:>14.2f}% {d_acc*100:>14.2f}% {(d_acc-b_acc)*100:>+11.2f}%")
print(f"{'Precision':<25} {b_prec*100:>14.2f}% {d_prec*100:>14.2f}% {(d_prec-b_prec)*100:>+11.2f}%")
print(f"{'Recall':<25} {b_rec*100:>14.2f}% {d_rec*100:>14.2f}% {(d_rec-b_rec)*100:>+11.2f}%")
print(f"{'F1-Score':<25} {b_f1*100:>14.2f}% {d_f1*100:>14.2f}% {(d_f1-b_f1)*100:>+11.2f}%")
print(f"{'Overall IoU':<25} {b_iou*100:>14.2f}% {d_iou*100:>14.2f}% {(d_iou-b_iou)*100:>+11.2f}%")
print(f"{'Average IoU (per image)':<25} {b_avg_iou*100:>14.2f}% {d_avg_iou*100:>14.2f}% {(d_avg_iou-b_avg_iou)*100:>+11.2f}%")

print("\n🌟 THIN CLOUD DETECTION (Primary Metric):")
print("-"*70)
print(f"{'Thin Cloud Recall':<25} {b_thin_recall*100:>14.2f}% {d_thin_recall*100:>14.2f}% {(d_thin_recall-b_thin_recall)*100:>+11.2f}%")
print(f"{'Thin Cloud IoU (avg)':<25} {b_avg_thin_iou*100:>14.2f}% {d_avg_thin_iou*100:>14.2f}% {(d_avg_thin_iou-b_avg_thin_iou)*100:>+11.2f}%")

print("\n" + "="*70)
print("✅ EVALUATION COMPLETE!")
print("="*70)

## 8. Summary Table for Thesis

In [ ]:
print("\n📋 COPY THIS TO YOUR THESIS (Table 4.3.x):")
print("="*60)
print()
print("| Metric | CNN Baseline | DQN (100k) | Change |")
print("|--------|--------------|------------|--------|")
print(f"| Accuracy | {b_acc*100:.2f}% | {d_acc*100:.2f}% | {(d_acc-b_acc)*100:+.2f}% |")
print(f"| Precision | {b_prec*100:.2f}% | {d_prec*100:.2f}% | {(d_prec-b_prec)*100:+.2f}% |")
print(f"| Recall | {b_rec*100:.2f}% | {d_rec*100:.2f}% | {(d_rec-b_rec)*100:+.2f}% |")
print(f"| F1-Score | {b_f1*100:.2f}% | {d_f1*100:.2f}% | {(d_f1-b_f1)*100:+.2f}% |")
print(f"| **Overall IoU** | {b_iou*100:.2f}% | {d_iou*100:.2f}% | **{(d_iou-b_iou)*100:+.2f}%** |")
print(f"| **Thin Cloud Recall** | {b_thin_recall*100:.2f}% | {d_thin_recall*100:.2f}% | **{(d_thin_recall-b_thin_recall)*100:+.2f}%** |")
print(f"| Thin Cloud IoU | {b_avg_thin_iou*100:.2f}% | {d_avg_thin_iou*100:.2f}% | {(d_avg_thin_iou-b_avg_thin_iou)*100:+.2f}% |")
print()
print("="*60)